# ContractBuilder Node — Unit Tests
Step-by-step testing of the ContractBuilder Node.
Run each cell independently to inspect the generated ServiceContract and logs.

The ContractBuilder makes **2 real LLM calls** (locator + extractor) and uses RAG:
- **queries** the persistent 3GPP corpus collection (`3gpp_rel18_all`) to ground the
  contract — if it has not been indexed, it degrades gracefully (no corpus context);
- **indexes** the document's sections into a temporary `spec_tmp_*` collection for the
  Extractor/Reflector. This notebook drops that temporary collection at the end.

Requires a running Qdrant on localhost:6333.

In [ ]:
# Step 1 — Imports and setup
# Run this cell first before any other cell.

from openapi_rulesbank.config import get_logger
from openapi_rulesbank.config.paths import TSPEC_DATA_TEST_FILE
from openapi_rulesbank.nodes.reader import reader_node
from openapi_rulesbank.nodes.contract_builder import contract_builder_node, _build_sections_summary
from openapi_rulesbank.schemas.contract import ServiceContract, format_contract_summary

# The service the contract is built for, taken from settings.TARGET_YAML — the
# single switch for local testing. Change it there to target another MnS in the
# same document; outputs follow into that service's own folder.
from openapi_rulesbank.config.settings import TARGET_YAML
TARGET_SERVICE = TARGET_YAML

logger = get_logger(__name__)
logger.info(f'Imports OK — target service: {TARGET_SERVICE}')

In [2]:
# Step 2 — Run Reader Node to prepare state
# The ContractBuilder depends on parsed_sections, so we run the Reader first.
# This is the same state that LangGraph would pass between the two nodes.

reader_state = {
    'main_doc_path':       TSPEC_DATA_TEST_FILE,
    'auxiliary_doc_paths': [],
    'target_yaml':         TARGET_SERVICE,
}

reader_result = reader_node(reader_state)

assert len(reader_result['parsed_sections']) > 0, 'Reader returned no sections'

logger.info(f"Reader OK — {len(reader_result['parsed_sections'])} section(s) ready for ContractBuilder.")

2026-07-21 00:15:13 [INFO] openapi_rulesbank.nodes.reader: Reader Node started.
2026-07-21 00:15:14 [INFO] openapi_rulesbank.nodes.reader: Reader sections: 726 total = 725 for planner + 1 excluded by reader
2026-07-21 00:15:14 [DEBUG] openapi_rulesbank.nodes.reader: Loaded 0 auxiliary document(s).
2026-07-21 00:15:14 [DEBUG] openapi_rulesbank.tools.document_tools: Loaded OpenAPI reference v3.0.1 from '/home/arimatea/Documents/Pessoal/Mestrado/0-Mestrado_Unicamp_2025/5-Projeto_mestrado_ericsson/openapi_multiagents/workspace/openapi_rulesbank/src/openapi_rulesbank/config/../../../data/references/openapi_reference/openapi_spec_3.0.1.md'.
2026-07-21 00:15:14 [INFO] openapi_rulesbank.nodes.reader: Reader Node complete.
2026-07-21 00:15:14 [INFO] __main__: Reader OK — 725 section(s) ready for ContractBuilder.


In [3]:
# Step 3 — Test _build_sections_summary()
# Verifies the helper formats sections into the locator table: section_id | title | preview

summary = _build_sections_summary(reader_result['parsed_sections'])

assert isinstance(summary, str) and len(summary) > 0, 'Expected non-empty summary string'

lines = summary.splitlines()
assert all('|' in l for l in lines[:5]), 'Expected id | title | preview format in each line'

logger.info(f'_build_sections_summary OK — {len(lines)} lines generated.')
logger.info('Preview (first 3 lines):')
for line in lines[:3]:
    logger.info(f'  {line[:120]}')

2026-07-21 00:15:14 [INFO] __main__: _build_sections_summary OK — 725 lines generated.
2026-07-21 00:15:14 [INFO] __main__: Preview (first 3 lines):
2026-07-21 00:15:14 [INFO] __main__:   1 | 3.1 Definitions | For the purposes of the present document, the terms and definitions given in TR 21.905 \[1\] and t
2026-07-21 00:15:14 [INFO] __main__:   2 | 3.2 Abbreviations | For the purposes of the present document, the abbreviations given in TR 21.905 \[1\] and the fol
2026-07-21 00:15:14 [INFO] __main__:   3 | 11.1 Generic provisioning management service | #


In [4]:
# Step 3b — Check whether the 3GPP corpus RAG is available
# The ContractBuilder queries GPP_CORPUS_COLLECTION to ground the contract. If it
# has not been indexed (run qdrant_3gpp_rag/scripts/index_3gpp_all.py), the node
# degrades gracefully and builds the contract without corpus context. This cell
# just reports which mode the run below will use.

from openapi_rulesbank.config.settings import GPP_CORPUS_COLLECTION
from openapi_rulesbank.tools.rag_tools import _spec_factory, search_3gpp_corpus

corpus_exists = _spec_factory().client.collection_exists(GPP_CORPUS_COLLECTION)
logger.info(f"3GPP corpus collection '{GPP_CORPUS_COLLECTION}' present? {corpus_exists}")

if corpus_exists:
    probe = search_3gpp_corpus(f'{TARGET_SERVICE} management service purpose capabilities')
    logger.info(f'  corpus grounding ACTIVE — retrieved {len(probe)} chars of context.')
    logger.info(f'  preview: {probe[:160]!r}')
else:
    logger.warning('  corpus grounding INACTIVE — contract will be built without corpus context.')
    logger.warning('  build it with: qdrant_3gpp_rag/scripts/index_3gpp_all.py --collection '
                   f'{GPP_CORPUS_COLLECTION}')

2026-07-21 00:15:14 [INFO] qdrant_3gpp_rag.factory: Initializing QdrantFactory (host=localhost, port=6333, model=sentence-transformers/all-MiniLM-L6-v2, device=cuda)
2026-07-21 00:15:14 [INFO] sentence_transformers.SentenceTransformer: Load pretrained SentenceTransformer: sentence-transformers/all-MiniLM-L6-v2
2026-07-21 00:15:20 [WARNING] huggingface_hub.utils._http: Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4135.60it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
2026-07-21 00:15:25 [INFO] __main__: 3GPP corpus collection '3gpp_rel18_all' present? False
2026-07-21 00

In [5]:
# Step 4 — Run contract_builder_node() end-to-end
# Makes 2 real LLM API calls (locator + extractor). Ensure OPENAI_API_KEY is set in .env.
# Also queries the 3GPP corpus RAG and indexes the spec into a temporary collection.

contract_state = {
    'parsed_sections': reader_result['parsed_sections'],
    'target_yaml':     TARGET_SERVICE,
}

contract_result = contract_builder_node(contract_state)

assert 'service_contract' in contract_result
contract = contract_result['service_contract']

# The temporary spec collection the node created (dropped in Step 7).
spec_collection = contract_result.get('spec_collection_name', '')

# Round-trips through the schema — raises if any field has the wrong shape/type.
ServiceContract(**contract)
assert contract['service_name'] == TARGET_SERVICE

oos = contract['out_of_scope_signals']

# The contract must scope the target on two axes:
#  (1) stage 3 block identified (so the Planner extracts only from the solution set),
#  (2) out-of-scope signals contain BOTH the other MnS (named services) AND at
#      least one out-of-scope section block (pattern "<clause>.x (out of scope)").
assert contract['stage_scope'].strip(), 'stage_scope is empty — locator did not find the stage 3 block'
assert any('(out of scope)' in x and 'service' in x.lower() for x in oos), \
    'No other Management Service flagged as out of scope'
assert any(x.split(' ')[0].endswith('.x') for x in oos), \
    'No out-of-scope section block (e.g. "11.1.x (out of scope)") recorded'

logger.info('contract_builder_node OK')
logger.info(f"  capabilities        : {len(contract['capabilities'])}")
logger.info(f"  key_concepts        : {len(contract['key_concepts'])}")
logger.info(f"  inclusion_keywords  : {len(contract['inclusion_keywords'])}")
logger.info(f"  exclusion_keywords  : {len(contract['exclusion_keywords'])}")
logger.info(f"  out_of_scope        : {len(oos)} signal(s)")
logger.info(f"  stage_scope set     : {bool(contract['stage_scope'].strip())}")
logger.info(f"  spec collection     : {spec_collection or '(none)'}")

2026-07-21 00:15:25 [INFO] openapi_rulesbank.nodes.contract_builder: ContractBuilder Node started.
2026-07-21 00:15:36 [INFO] openapi_rulesbank.nodes.contract_builder: ContractBuilder locator: purpose=['4'], capabilities=['5', '6', '7', '8', '9', '10', '11', '12', '13', '14', '15', '16', '17', '18', '19', '20', '21', '22', '23', '24', '25', '28', '29', '30', '31', '32', '33', '34', '35', '36', '37', '38', '39', '40', '41', '42', '43', '44', '45', '46', '47', '48', '49', '50', '51', '52', '53', '54'], concepts=['52', '53', '54']
2026-07-21 00:15:36 [WARNING] openapi_rulesbank.tools.rag_tools: 3GPP corpus collection '3gpp_rel18_all' not found — run qdrant_3gpp_rag/scripts/index_3gpp_all.py to build it. Continuing without corpus RAG.
2026-07-21 00:15:45 [INFO] openapi_rulesbank.nodes.contract_builder: ContractBuilder complete: 9 capabilities, 20 inclusion keyword(s), 0 exclusion keyword(s).
2026-07-21 00:15:45 [INFO] qdrant_3gpp_rag.indexer: Chunked 725 segment(s) into 1862 chunk(s) for '

In [6]:
# Step 5 — Inspect the generated contract (formatted summary)
# This is the exact block the Planner and Extractor receive as scope context.

print(format_contract_summary(contract))

SERVICE CONTRACT
Service: ProvMnS
Purpose: This service exists to provide generic provisioning management services for managing managed objects in the Network Resource Model (NRM), including creation, modification, deletion, retrieval, and notifications of managed object changes.
Capabilities: createMOI, modifyMOIAttributes, deleteMOI, getMOIAttributes, notifyMOICreation, notifyMOIDeletion, notifyMOIAttributeValueChanges, notifyMOIChanges, notifyEvent
Key concepts: Managed Object Instance (MOI), ManagedEntity, NRM updates, attribute, attribute name, attribute value, attribute field, attribute element, attribute field element, simple type, complex type, structured attribute, multi-valued attribute, notificationId, correlatedNotifications, operation, path, value, oldValue, sourceIndicator, scope, scopeType, scopeLevel, filter, modificationList
Inclusion keywords: Managed Object Instance, MOI, ManagedEntity, notifyMOICreation, notifyMOIDeletion, notifyMOIAttributeValueChanges, notifyMOICh

In [7]:
# Step 6 — Inspect capabilities and scope in detail
# Each capability: name, type (operation|notification|state|subscription|other), description.

for i, cap in enumerate(contract['capabilities']):
    print(f"[{i}] ({cap['type']}) {cap['name']}")
    print(f"     {cap['description']}")

print(f"\nKey concepts       : {contract['key_concepts']}")
print(f"Inclusion keywords : {contract['inclusion_keywords']}")
print(f"Exclusion keywords : {contract['exclusion_keywords']}")
print(f"Out-of-scope       : {contract['out_of_scope_signals']}")

# Stage 2 vs stage 3 scope — tells the Planner/Extractor to extract ONLY from
# the target's stage 3 block (the RESTful HTTP solution set).
print(f"\nStage scope        :\n  {contract['stage_scope']}")

[0] (operation) createMOI
     This operation is invoked by MnS consumer to request the MnS producer to create a Managed Object instance in the MIB maintained by MnS producer.
[1] (operation) modifyMOIAttributes
     This operation is invoked by MnS consumer to request the modification of one or more Managed Object instances from MnS producer.
[2] (operation) deleteMOI
     This operation is invoked by MnS consumer to request the deletion of one or more Managed Object instances in the MIB maintained by the MnS producer.
[3] (operation) getMOIAttributes
     This operation is invoked by MnS consumer to request the retrieval of management information (Managed Object attribute names and values) from the MIB maintained by MnS producer.
[4] (notification) notifyMOICreation
     This notification notifies the subscribed consumers that a new Managed Object Instance has been created.
[5] (notification) notifyMOIDeletion
     This notification notifies the subscribed consumers that an existing 

In [ ]:
# Step 7 — Save the full contract JSON, then drop the temporary spec collection
# The JSON goes to data/outputs/contracts/<Service>/ (one folder per MnS, so runs
# for different services stay separate); the spec_tmp_* collection is dropped.

import json
import os
from datetime import datetime
from openapi_rulesbank.config.paths import contracts_dir_for
from openapi_rulesbank.tools.rag_tools import drop_target_spec_collection

contracts_dir = contracts_dir_for(TARGET_SERVICE)
os.makedirs(contracts_dir, exist_ok=True)

timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
out_path = os.path.join(contracts_dir, f'contract_{TARGET_SERVICE}_{timestamp}.json')

with open(out_path, 'w', encoding='utf-8') as f:
    json.dump(contract, f, indent=2, ensure_ascii=False)

logger.info(f'Contract saved to: {out_path}')

# Clean up the temporary spec collection created by the node.
if spec_collection:
    drop_target_spec_collection(spec_collection)
    logger.info(f'Dropped temporary spec collection: {spec_collection}')